# Lab 4: Hypothesis Testing

> **Course:** Data Science Foundations
> **Lesson:** Day 4 Lab, companion to `4-hypothesis-testing.md`
> **Dataset:** `penguins` (seaborn) plus simulated A/B experiments
> **Estimated time:** 2 hours (Parts 5 and 7 are extensions and can be skipped if time is short)

## Lab Objectives

- Frame real questions as $H_0$ / $H_1$ and pick an appropriate test
- Run a one-sample t-test **by hand** and confirm it with SciPy, including a **confidence interval**
- Run a **Welch** two-sample t-test, report an **effect size** (Cohen's d), and check the assumptions
- Use chi-square tests of independence and read the expected-count table
- Build a **permutation test** from scratch
- Show by simulation why **multiple testing** inflates false positives, and how Bonferroni fixes it
- Estimate the **sample size** an A/B test needs for 80% power

**How to work through this lab**

- Cells marked `# TODO` are yours to complete. Replace each `None` or comment with working code.
- Run every **Checkpoint** cell. If it prints `passed`, move on. If it raises an error, fix your code first.
- Each **Question** needs a short written answer in the "Your answer" cell below it.
- The dataset is different from the lesson on purpose. Adapt the lesson code; don't paste it.
- Worked solutions are in the `solutions/` folder. Attempt every part yourself before you open them: the struggle is where the learning happens.

## Part 0: Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)
alpha = 0.05

penguins = sns.load_dataset("penguins").dropna().reset_index(drop=True)
print(f"Shape: {penguins.shape}")
penguins.head()

## Part 1: Framing the Question

For each scenario write: $H_0$, $H_1$, one- or two-tailed, the test you would use, and which error (Type I or II) is more costly.

1. A bakery claims its loaves weigh 500 g on average. A consumer group weighs 30 loaves.
2. An online shop tests a new checkout page against the old one and records whether each visitor buys.
3. A logistics firm compares average delivery times across **three** couriers.
4. A bank asks whether customers' preferred payment method (card, mobile money, cash) is related to their region.

**Your answer:**

*Write your answer here.*

## Part 2: One-Sample t-Test by Hand

A field guide states that Adelie penguins have a mean flipper length of **190 mm**. Does our sample disagree?

$$t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}}, \qquad df = n - 1, \qquad \text{95\% CI} = \bar{x} \pm t_{0.975,\,df} \cdot \frac{s}{\sqrt{n}}$$

In [ ]:
adelie_flipper = penguins.loc[penguins["species"] == "Adelie", "flipper_length_mm"]
mu_0 = 190

# TODO 2.1: compute each piece by hand
n = None
x_bar = None
s = None
se = None
t_manual = None
df = None
p_manual = None        # two-tailed: 2 * stats.t.sf(abs(t_manual), df)
ci_manual = None       # (lower, upper), using stats.t.ppf(0.975, df)

# TODO 2.2: the same test with stats.ttest_1samp
res = None

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert abs(t_manual - res.statistic) < 1e-9 and abs(p_manual - res.pvalue) < 1e-9
ci = res.confidence_interval(0.95)
assert abs(ci_manual[0] - ci.low) < 1e-9 and abs(ci_manual[1] - ci.high) < 1e-9
print("passed")

**Question 2.** State your decision. Does it *prove* that the true mean is 190 mm? Re-run with `mu_0 = 192`. What changes, and how could you have predicted it from the confidence interval alone?

**Your answer:**

*Write your answer here.*

## Part 3: Welch Two-Sample t-Test, Effect Size, and Assumptions

**Question:** do male and female Gentoo penguins differ in body mass?

As the lesson explains, the t-statistic with separate variances, $\sqrt{s_A^2/n_A + s_B^2/n_B}$, is **Welch's** t-test. Remember that `stats.ttest_ind` defaults to Student's version, which assumes equal variances, so pass `equal_var=False`. Here you will also run Student's version to see when the two differ.

**Cohen's d** measures the size of the difference in standard-deviation units: $d = (\bar{x}_A - \bar{x}_B) / s_{pooled}$, where $s_{pooled} = \sqrt{\frac{(n_A - 1)s_A^2 + (n_B - 1)s_B^2}{n_A + n_B - 2}}$. As a rough guide, 0.2 is small, 0.5 medium, and 0.8 large.

In [ ]:
gentoo = penguins[penguins["species"] == "Gentoo"]
male = gentoo.loc[gentoo["sex"] == "Male", "body_mass_g"]
female = gentoo.loc[gentoo["sex"] == "Female", "body_mass_g"]

# TODO 3.1: Welch t-test (equal_var=False). Also run Student's version to compare the p-values.
welch = None
student = None

# TODO 3.2: Cohen's d
def cohens_d(a, b):
    return None

# TODO 3.3: a 95% CI for the difference in means (hint: welch.confidence_interval(0.95))
diff_ci = None

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert welch.pvalue < 1e-10
assert 2 < cohens_d(male, female) < 4
print("passed")

In [ ]:
# TODO 3.4: check the Normality assumption. Plot histograms of each group,
# run stats.shapiro on each, and run the non-parametric stats.mannwhitneyu as a robustness check.

**Question 3.** Report the result the way you would in a short write-up: direction, size in grams, CI, effect size, p-value, and whether the assumptions look reasonable.

**Your answer:**

*Write your answer here.*

## Part 4: Chi-Square Tests of Independence

The test compares observed counts with the counts **expected if the variables were independent**: $E_{ij} = \frac{\text{row total}_i \times \text{column total}_j}{N}$. It is only reliable when every expected count is at least about 5.

In [ ]:
# TODO 4.1: is species independent of island? Show the observed table, the expected table, chi2, dof and p.

# TODO 4.2: is sex independent of species? Same steps.

**Question 4.** Interpret both tests. For species vs island, what do the zeros in the observed table tell you that the p-value does not?

**Your answer:**

*Write your answer here.*

## Part 5: More Than Two Groups: ANOVA (extension)

To compare three or more group means, use a one-way **ANOVA** (`stats.f_oneway`) instead of many t-tests. A significant ANOVA says *at least one* mean differs. Follow-up pairwise tests show which ones, but they need a multiple-testing correction (Part 7).

In [ ]:
# TODO 5.1: ANOVA of bill_length_mm across the three species; plot a box plot too

# TODO 5.2: pairwise Welch t-tests for the 3 species pairs; mark each significant at alpha / 3 (Bonferroni)

**Your answer:**

*Write your answer here.*

## Part 6: A Permutation Test from Scratch

A permutation test builds the null distribution directly, with no Normality assumption. **If $H_0$ is true, the group labels are arbitrary.** So shuffle the labels many times, recompute the difference each time, and see how often a shuffled difference is at least as extreme as the real one.

Question: do male and female **Chinstrap** penguins differ in bill depth?

In [ ]:
chin = penguins[penguins["species"] == "Chinstrap"]
a = chin.loc[chin["sex"] == "Male", "bill_depth_mm"].to_numpy()
b = chin.loc[chin["sex"] == "Female", "bill_depth_mm"].to_numpy()


def permutation_test(a, b, n_perm=10_000, rng=rng):
    # TODO: return (observed_diff, null_diffs, p_value)
    # observed_diff = mean(a) - mean(b)
    # for each permutation: shuffle the pooled values, split into groups of len(a) and len(b), store the mean difference
    # two-sided p_value = (count of |null| >= |observed| + 1) / (n_perm + 1)
    return None


# TODO: run it, plot the histogram of null_diffs with a vertical line at the observed difference,
# and compare the p-value with a Welch t-test

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
obs, null, p = permutation_test(np.array([1.0, 2, 3]), np.array([1.0, 2, 3]), n_perm=200)
assert obs == 0 and p == 1.0 and len(null) == 200
print("passed")

**Your answer:**

*Write your answer here.*

## Part 7: The Multiple Testing Trap (extension)

Your team runs an A/A test (both groups get the *same* page, so $H_0$ is true for everything) and checks **20 different metrics**, each at α = 0.05. How often does *at least one* metric come out "significant"?

In [ ]:
n_experiments, n_metrics, n_per_group = 1000, 20, 50

# TODO 7.1: simulate data of shape (n_experiments, n_metrics, n_per_group) for each group from the SAME Normal,
# then compute all p-values in one call: stats.ttest_ind(group_a, group_b, axis=-1).pvalue  -> shape (1000, 20)
p_values = None

# TODO 7.2: share of experiments with at least one p <= alpha, compared with 1 - 0.95**20
# TODO 7.3: the same share using the Bonferroni threshold alpha / n_metrics

**Your answer:**

*Write your answer here.*

## Part 8: How Big Should the A/B Test Be?

The lesson's simulation found that 1,000 users per group gives only modest power to detect a lift from 10% to 12%. Find the sample size per group that gives **80% power**.

Planning formula for comparison: $n \approx \dfrac{(z_{1-\alpha/2} + z_{0.8})^2 \,[p_A(1-p_A) + p_B(1-p_B)]}{(p_B - p_A)^2}$

In [ ]:
p_A, p_B, n_sims = 0.10, 0.12, 4000
sizes = [500, 1000, 2000, 3000, 4000, 5000, 6000]


def simulated_power(n, p_A, p_B, n_sims, rng=rng):
    # TODO: simulate n_sims experiments at once:
    #   conv_A = rng.binomial(n, p_A, n_sims), conv_B = rng.binomial(n, p_B, n_sims)
    # compute the two-proportion z statistic and two-tailed p-value for each (vectorised, no loop),
    # and return the share of experiments with p <= alpha
    return None


# TODO: power for each size in `sizes`, a plot of power vs n with a line at 0.8, and the formula's n

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert simulated_power(1000, 0.10, 0.10, 4000) < 0.08   # with no real effect, "power" is just alpha
assert simulated_power(6000, 0.10, 0.12, 4000) > 0.8
print("passed")

**Question 8.** A manager wants to run the test for one week with 1,000 users per group "to be quick". Using your results, explain the risk in two sentences.

**Your answer:**

*Write your answer here.*

## Part 9: Wrap-Up

Name three things from today that the p-value alone would not have told you.

**Your answer:**

*Write your answer here.*

> Submit your completed notebook to the Kanban board under your name by end of day.